# GRSL final run — level vs spatial skill in nocturnal urban LST

**One notebook, top to bottom.** Run every cell in order. Section 5 does all the training and saves
every prediction to Drive; everything after it only reads those files, so if the analysis needs
changing you never retrain.

| Section | What it does | Retrains? |
|---|---|---|
| 1 Setup | paths, configs, hyperparameters, **run plan** | – |
| 2 Imports | libraries, seeding, kNN graph | – |
| 3 Data | loading, node filter, spatial blocks, night hold-out | – |
| 4 Models | random forest, GAT (with the loss-mask bug fixed) | – |
| 5 **Run** | trains every model in the run plan, saves predictions. **Resumable.** | yes |
| 6 Metrics | decomposition, oracle, spread ratio, bootstrap by night | no |
| 7 Results | Table I, decoupling, ablation, GAT diagnostics, per-sensor | no |
| 8 Data checks | the open doubts: correlations, sds, persistence, counts, Moran's I, NO₂ | no |
| 9 Figure | decoupling figure (anomaly gap in °C, 95% bootstrap bars) | no |
| 10 Paper numbers | every number the paper quotes, in one printout | no |

**What changed from `main_fixed_v3`, and why**

1. **Loss bug fixed.** The old `composite_loss` overwrote the masked loss with an unmasked one, so the
   GAT trained on every node's target, including test blocks and interpolated values. It now uses only
   observed cells of the training split.
2. **Tree = random forest + spatial lag, everywhere.** No RF-vs-GBR pick on test RMSE. The bootstrap,
   Table I and the figure now all use the same model.
3. **Two split designs.** `spatial` (unseen locations, all nights; the old design, which is gap-filling) and
   `spatiotemporal` (unseen locations *and* unseen nights, held out in chunks of consecutive nights).
   Answers referee comment 1.
4. **Four spatial folds** (rotating test blocks). Answers the spatial-uncertainty half of comment 8.
5. **Bootstrap resamples calendar nights** (a night's Terra and Aqua passes move together), 95% intervals.
6. **Ablation reported with the decomposition**, and relabelled "static surface predictors" (NDVI, NDBI,
   building height, building cover, NO₂). Comment 6.
7. **GAT diagnostics**: PairNorm off, physics prior off, and level error against 2 m temperature. Comment 3.
8. **Spread ratio reported with S/r.** Comment 7.
9. **Data checks** for every open doubt (section 8).

**Order of operations:** `SMOKE_TEST` starts as `True`. Run all cells once (about 5 minutes on one
config, 3 epochs) to catch path or column problems. Then set `SMOKE_TEST = False` and run all cells
again for the real run. Smoke outputs go to a separate `_smoke` folder, so they never mix with the
real results. Runtime GPU: T4 is fine.

**Runtime:** the run plan prints an estimate. Everything is saved per model, so if Colab disconnects,
re-run from the top and it resumes where it stopped. Stage A (fold 0, every model) finishes first
and is enough for a complete paper; Stage B adds folds 1–3.

In [ ]:
# ============================================================
# 1. SETUP
# ============================================================
import os, sys, shutil, json, time

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')

# ---- paths (env vars override, for running outside Colab) ----
DRIVE_DATA = os.environ.get('GRSL_DRIVE_DATA', '/content/drive/MyDrive/UHI')
LOCAL_DATA = os.environ.get('GRSL_LOCAL_DATA', '/content/data_modis')
OUT_ROOT   = os.environ.get('GRSL_OUT_ROOT',   '/content/drive/MyDrive/UHI/PhyRes_MODIS')

SMOKE_TEST = True           # FIRST RUN: True (one config, fold 0, 3 epochs, ~5 min). Then set False.
RUN_TAG    = 'final_v1' + ('_smoke' if SMOKE_TEST else '')
OUT        = f'{OUT_ROOT}/{RUN_TAG}'
PRED_DIR, CTX_DIR, RES_DIR = f'{OUT}/preds', f'{OUT}/context', f'{OUT}/results'
for d in (PRED_DIR, CTX_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

DATASETS = [
    ('Bangalore', 'Blr', 'April',    'Blr_April_MODIS_60d.csv',    'Blr_Nodes_1km.geojson'),
    ('Bangalore', 'Blr', 'December', 'Blr_December_MODIS_60d.csv', 'Blr_Nodes_1km.geojson'),
    ('Hyderabad', 'Hyd', 'April',    'Hyd_April_MODIS_60d.csv',    'Hyd_Nodes_1km.geojson'),
    ('Hyderabad', 'Hyd', 'December', 'Hyd_December_MODIS_60d.csv', 'Hyd_Nodes_1km.geojson'),
    ('Delhi',     'Dlh', 'April',    'Dlh_April_MODIS_60d.csv',    'Dlh_Nodes_1km.geojson'),
    ('Delhi',     'Dlh', 'December', 'Dlh_December_MODIS_60d.csv', 'Dlh_Nodes_1km.geojson'),
]
if SMOKE_TEST:
    DATASETS = DATASETS[:1]

os.makedirs(LOCAL_DATA, exist_ok=True)
missing = []
for name in sorted({f for d in DATASETS for f in d[3:5]}):
    src, dst = f'{DRIVE_DATA}/{name}', f'{LOCAL_DATA}/{name}'
    if os.path.exists(dst) and (not os.path.exists(src) or
                                os.path.getsize(dst) == os.path.getsize(src)):
        continue
    if not os.path.exists(src):
        missing.append(name); continue
    shutil.copy2(src, dst)
if missing:
    raise FileNotFoundError(f'missing on Drive: {missing}')

CONFIGS = [dict(city=c, short=s, month=m, label=f'{s}-{"Pre" if m == "April" else "Post"}',
                csv=f'{LOCAL_DATA}/{csv}', geojson=f'{LOCAL_DATA}/{geo}')
           for c, s, m, csv, geo in DATASETS]

# ---- data handling (unchanged from the paper's pipeline) ----
FEATURE_COLS = ['t2m', 'u10', 'v10', 'ssr', 'str', 'slhf',
                'NDVI', 'NDBI', 'bld_height', 'bld_cover', 'traffic_no2_proxy']
ATMOS_IDX  = [0, 1, 2, 3, 4, 5]
STATIC_IDX = [6, 7, 8, 9, 10]      # "static surface predictors" in the paper
TARGET_COL, TIME_COL, ID_COL = 'lst_modis', 'timestamp', 'id'
TARGET_CRS = 32643
NIGHT_ONLY, QC_STRICT = True, False
MIN_NODE_COVERAGE = 0.60           # node kept if observed on >= 60% of passes
MIN_NODES_PER_PASS = 5             # a pass is scored only with >= 5 observed test nodes
UTC_OFFSET_H = 5.5                 # IST; used only to group passes into calendar nights

# ---- splits ----
SPLIT = dict(n_blocks=5, frac_test=0.25, frac_val=0.15,
             night_chunk=4,          # spatiotemporal: nights held out in runs of 4 consecutive nights
             seed=10)
FOLDS = [0] if SMOKE_TEST else [0, 1, 2, 3]
MODES = ['spatial', 'spatiotemporal']

# ---- hyperparameters ----
# GAT: fixed a priori, not tuned (state this in the paper). Same values as the original sweep.
HP = dict(window_size=5, k_neighbors=8, spatial_dim=32, lstm_hidden=64, dropout=0.2,
          epochs=3 if SMOKE_TEST else 200, patience=40, lr=8e-4, weight_decay=1e-4, clip=0.5,
          alpha=1.0, gamma=0.03, sparsity_w=0.005, huber_delta=1.5, seed=10)
# Push-pull prior thresholds (raw units), as in the original sweep
COVER_T, NDVI_T, PUSH_MARGIN_C, PULL_MARGIN_C = 0.30, 0.40, 1.0, 0.5
# RF: small grid, selected on validation cells only
RF = dict(n_estimators=40 if SMOKE_TEST else 300, leaf_grid=(1, 2, 8), max_features='sqrt',
          lag_k=8)

# ---- models ----
MODELS = {
    'RF_lag':         dict(kind='rf',  drop=[],         lag=True),
    'RF_plain':       dict(kind='rf',  drop=[],         lag=False),
    'RF_noStatic':    dict(kind='rf',  drop=STATIC_IDX, lag=False),
    'GAT':            dict(kind='gat', drop=[],         pairnorm=True,  prior=True),
    'GAT_noStatic':   dict(kind='gat', drop=STATIC_IDX, pairnorm=True,  prior=True),
    'GAT_noPairNorm': dict(kind='gat', drop=[],         pairnorm=False, prior=True),
    'GAT_noPrior':    dict(kind='gat', drop=[],         pairnorm=True,  prior=False),
}
TREE, GRAPH = 'RF_lag', 'GAT'      # the two models the paper compares

# ---- run plan: (models, modes, folds). Stage A first: a complete paper on fold 0. ----
RUN_PLAN = [
    ('A', list(MODELS),                     MODES, [0]),
    ('B', ['RF_lag', 'RF_plain', 'RF_noStatic', 'GAT'], MODES, [f for f in FOLDS if f != 0]),
]
GAT_MIN_PER_RUN = 5.0              # rough Colab T4 estimate, used only for the ETA printout
n_gat = sum(sum(MODELS[m]['kind'] == 'gat' for m in ms) * len(md) * len(fs)
            for _, ms, md, fs in RUN_PLAN) * len(CONFIGS)
print(f'{len(CONFIGS)} configs | modes {MODES} | folds {FOLDS}')
print(f'GAT trainings planned: {n_gat}  (~{n_gat * GAT_MIN_PER_RUN / 60:.1f} h at '
      f'{GAT_MIN_PER_RUN:.0f} min each, plus trees)')
print(f'outputs -> {OUT}')
print(f'SMOKE_TEST = {SMOKE_TEST}')

In [ ]:
# ============================================================
# 2. IMPORTS, SEEDING, kNN GRAPH
# ============================================================
import random, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch_geometric.nn import GATConv, PairNorm
from torch_geometric.utils import dropout_edge
from sklearn.ensemble import RandomForestRegressor
from scipy import stats

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, torch.cuda.get_device_name(0) if device.type == 'cuda' else '')


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True   # GPU runs are not bit-reproducible; say so in the paper


def knn_indices(pos, k):
    """(N, k) indices of the k nearest other nodes, Euclidean (pos in metres)."""
    d2 = ((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1)
    np.fill_diagonal(d2, np.inf)
    return np.argsort(d2, axis=1)[:, :k]


def knn_edge_index(pos, k):
    """edge_index [2, N*k], row 0 = neighbour (source), row 1 = node (target)."""
    nbr = knn_indices(pos, k)
    src = nbr.reshape(-1)
    dst = np.repeat(np.arange(len(pos)), k)
    return np.stack([src, dst]).astype(np.int64)

In [ ]:
# ============================================================
# 3. DATA
# ------------------------------------------------------------
# load_raw()   : everything that does not depend on the split (cached).
# make_split() : blocks, fold, night hold-out, normalisation, cell masks.
#
# Terms used below:
#   pass   = one MODIS night overpass (Terra ~22:30 or Aqua ~01:30 local).
#   night  = calendar night; its Terra and Aqua passes share a night id.
#   window = 5 consecutive passes of predictors; the target is LST at the
#            window's LAST pass (estimation, not forecasting).
#   cell   = (node, window). All masks below are (N, n_windows).
# ============================================================
_RAW_CACHE = {}


def _align_ids(s):
    try:
        return s.astype('int64')
    except (ValueError, TypeError):
        return s.astype(str)


def load_nodes(path):
    g = gpd.read_file(path)
    b = g.total_bounds
    if max(abs(b[0]), abs(b[2]), abs(b[1]), abs(b[3])) > 180:   # already metres
        g = g.set_crs(TARGET_CRS, allow_override=True)
    else:
        if g.crs is None:
            g = g.set_crs(4326)
        g = g.to_crs(epsg=TARGET_CRS)
    return g


def _sensor_of(pass_names):
    s = ' '.join(map(str, pass_names)).lower()
    if 'terra' in s or 'mod11' in s:
        return 'Terra'
    if 'aqua' in s or 'myd11' in s:
        return 'Aqua'
    return 'unknown'


def load_raw(cfg):
    key = cfg['csv']
    if key in _RAW_CACHE:
        return _RAW_CACHE[key]
    df = pd.read_csv(cfg['csv'])
    gdf = load_nodes(cfg['geojson'])
    for col in [ID_COL, TIME_COL, TARGET_COL, 'pass'] + FEATURE_COLS:
        if col not in df.columns:
            raise KeyError(f"'{col}' missing from {os.path.basename(cfg['csv'])}")
    all_columns = list(df.columns)
    df[ID_COL] = _align_ids(df[ID_COL]); gdf[ID_COL] = _align_ids(gdf[ID_COL])
    if df[ID_COL].dtype != gdf[ID_COL].dtype:
        df[ID_COL] = df[ID_COL].astype(str); gdf[ID_COL] = gdf[ID_COL].astype(str)
    df[TIME_COL] = (pd.to_datetime(df[TIME_COL], unit='ms')
                    if pd.api.types.is_numeric_dtype(df[TIME_COL])
                    else pd.to_datetime(df[TIME_COL]))
    n_nodes_total = df[ID_COL].nunique()
    if NIGHT_ONLY:
        df = df[df['pass'].astype(str).str.contains('night', case=False, na=False)]
    qc_info = {c: float(df[c].mean()) for c in ('qc_ok', 'qc_strict') if c in df.columns}
    if QC_STRICT and 'qc_strict' in df.columns:
        df = df[df['qc_strict'] == 1]

    times = np.sort(df[TIME_COL].unique())
    obs_rows = df[df[TARGET_COL].notna()]
    ncov = obs_rows.groupby(ID_COL)[TIME_COL].nunique() / len(times)
    keep = ncov[ncov >= MIN_NODE_COVERAGE].index
    df = df[df[ID_COL].isin(keep)]
    gdf = gdf[gdf[ID_COL].isin(keep)].sort_values(ID_COL).reset_index(drop=True)
    node_ids = np.sort(df[ID_COL].unique())
    if len(gdf) != len(node_ids) or not (gdf[ID_COL].values == node_ids).all():
        raise ValueError('node id mismatch between CSV and geojson')

    N, T, Fn = len(node_ids), len(times), len(FEATURE_COLS)
    ni = {v: i for i, v in enumerate(node_ids)}
    ti = {v: i for i, v in enumerate(times)}
    r = df[ID_COL].map(ni).values
    c = df[TIME_COL].map(ti).values
    cube = np.full((N, T, Fn), np.nan, np.float32)
    ymat = np.full((N, T), np.nan, np.float32)
    cube[r, c, :] = df[FEATURE_COLS].values.astype(np.float32)
    ymat[r, c] = df[TARGET_COL].values.astype(np.float32)
    observed = ~np.isnan(ymat)

    def fill(a):
        out = a.copy(); idx = np.arange(a.shape[1])
        for i in range(a.shape[0]):
            m = ~np.isnan(out[i])
            if m.sum() == 0:
                raise ValueError(f'node {i} has no valid values')
            out[i] = np.interp(idx, idx[m], out[i][m])
        return out
    for f in range(Fn):
        cube[:, :, f] = fill(cube[:, :, f])
    y_filled = fill(ymat)

    # sensor and calendar night of each pass
    pass_of_t = df.groupby(TIME_COL)['pass'].agg(lambda s: list(s.unique()))
    sensor = np.array([_sensor_of(pass_of_t.get(t, [])) for t in times])
    local = pd.to_datetime(times) + pd.Timedelta(hours=UTC_OFFSET_H) - pd.Timedelta(hours=12)
    night_key = local.normalize()
    night_id = pd.factorize(night_key, sort=True)[0]
    night_day = (night_key - night_key.min()).days.values   # days since first night

    px = gdf.geometry.centroid.x.values; py = gdf.geometry.centroid.y.values
    pos = np.column_stack([px, py]).astype(np.float64)
    nb = SPLIT['n_blocks']
    bx = np.floor((px - px.min()) / (np.ptp(px) + 1e-9) * nb).clip(0, nb - 1)
    by = np.floor((py - py.min()) / (np.ptp(py) + 1e-9) * nb).clip(0, nb - 1)
    block = (bx * nb + by).astype(int)

    R = dict(cfg=cfg, node_ids=node_ids, times=times, sensor=sensor, night_id=night_id,
             night_day=night_day, cube=cube, ymat=ymat, y_filled=y_filled, observed=observed,
             pos=pos, block=block, n_nodes_total=n_nodes_total, qc_info=qc_info,
             all_columns=all_columns, df_night=df)
    _RAW_CACHE[key] = R
    print(f"  {cfg['label']}: {T} passes ({(sensor == 'Terra').sum()} Terra / "
          f"{(sensor == 'Aqua').sum()} Aqua / {(sensor == 'unknown').sum()} unknown), "
          f"{night_id.max() + 1} nights | nodes {n_nodes_total} -> {N} kept | "
          f"{(~observed).mean():.1%} of target cells interpolated")
    return R


def _assign(n_items, fold, rng_seed, frac_test, frac_val):
    """Permute items, take a rotating test slice, then val, rest train."""
    rng = np.random.default_rng(rng_seed)
    perm = rng.permutation(n_items)
    n_te = max(1, int(n_items * frac_test)); n_va = max(1, int(n_items * frac_val))
    te = np.roll(perm, -fold * n_te)[:n_te]
    te_set = set(te.tolist())
    rest = np.array([p for p in perm if p not in te_set])
    return te, rest[:n_va], rest[n_va:]


def make_split(R, mode, fold):
    """Masks, windows and normalisation for one (config, mode, fold)."""
    W = HP['window_size']
    N, T = R['ymat'].shape
    tgt = np.arange(W - 1, T)                   # target pass of each window
    n_win = len(tgt)

    ublocks = np.unique(R['block'])
    te_i, va_i, tr_i = _assign(len(ublocks), fold, SPLIT['seed'],
                               SPLIT['frac_test'], SPLIT['frac_val'])
    node_role = np.full(N, 'train', dtype=object)
    node_role[np.isin(R['block'], ublocks[va_i])] = 'val'
    node_role[np.isin(R['block'], ublocks[te_i])] = 'test'

    if mode == 'spatial':
        time_role = np.full(n_win, 'all', dtype=object)
    elif mode == 'spatiotemporal':
        n_nights = R['night_id'].max() + 1
        chunk = np.arange(n_nights) // SPLIT['night_chunk']
        n_chunks = chunk.max() + 1
        te_c, va_c, tr_c = _assign(n_chunks, fold, SPLIT['seed'] + 1,
                                   SPLIT['frac_test'], SPLIT['frac_val'])
        chunk_role = np.full(n_chunks, 'train', dtype=object)
        chunk_role[va_c] = 'val'; chunk_role[te_c] = 'test'
        time_role = chunk_role[chunk[R['night_id'][tgt]]]
    else:
        raise ValueError(mode)

    obs_t = R['observed'][:, tgt]               # (N, n_win)
    masks = {}
    for role in ('train', 'val', 'test'):
        tm = np.ones(n_win, bool) if mode == 'spatial' else (time_role == role)
        masks[role] = (node_role == role)[:, None] & tm[None, :] & obs_t

    # normalisation from TRAINING cells only
    trn = node_role == 'train'
    xs = R['cube'][trn]
    x_mean = xs.mean(axis=(0, 1), keepdims=True); x_std = xs.std(axis=(0, 1), keepdims=True)
    x_norm = ((R['cube'] - x_mean) / (x_std + 1e-6)).astype(np.float32)
    ytr = R['y_filled'][:, tgt][masks['train']]
    t_mean, t_std = float(ytr.mean()), float(ytr.std())

    return dict(mode=mode, fold=fold, W=W, tgt=tgt, n_win=n_win, node_role=node_role,
                time_role=time_role, masks=masks, x_norm=x_norm, t_mean=t_mean, t_std=t_std,
                n_blocks=(len(tr_i), len(va_i), len(te_i)),
                n_nodes=tuple(int((node_role == r).sum()) for r in ('train', 'val', 'test')))


def save_context(R, S):
    """Everything the analysis needs, so sections 6-10 never touch the CSVs."""
    cfg = R['cfg']
    path = f"{CTX_DIR}/{cfg['label']}_{S['mode']}_f{S['fold']}.npz"
    if os.path.exists(path):
        return path
    tgt = S['tgt']
    np.savez_compressed(
        path,
        y=R['y_filled'][:, tgt], observed=R['observed'][:, tgt],
        m_train=S['masks']['train'], m_val=S['masks']['val'], m_test=S['masks']['test'],
        node_role=S['node_role'].astype(str), time_role=S['time_role'].astype(str),
        sensor=R['sensor'][tgt], night_id=R['night_id'][tgt], night_day=R['night_day'][tgt],
        t2m_level=R['cube'][:, tgt, 0].mean(axis=0),        # raw 2 m temperature, per pass
        pos=R['pos'], block=R['block'],
        n_blocks=np.array(S['n_blocks']), n_nodes=np.array(S['n_nodes']))
    return path


def build_tree_design(R, S, drop, lag):
    """(N, n_win, D): the flattened 5-pass window, plus neighbour means of static channels."""
    W, tgt = S['W'], S['tgt']
    keep_f = [f for f in range(len(FEATURE_COLS)) if f not in drop]
    X = np.stack([S['x_norm'][:, t - W + 1:t + 1, :][:, :, keep_f] for t in tgt], axis=1)
    parts = [X.reshape(X.shape[0], X.shape[1], -1)]
    if lag:
        ch = [keep_f.index(f) for f in STATIC_IDX if f in keep_f]
        nbr = knn_indices(R['pos'], RF['lag_k'])
        lagx = X[:, :, :, ch][nbr].mean(axis=1)             # (N, n_win, W, len(ch))
        parts.append(lagx.reshape(X.shape[0], X.shape[1], -1))
    return np.concatenate(parts, axis=-1).astype(np.float32)

In [ ]:
# ============================================================
# 4. MODELS
# ------------------------------------------------------------
# GAT: same architecture as the original sweep (GAT+skip layer, LSTM over
# the 5-pass window, physics head on atmospherics + residual head on static
# predictors, wind-pruned kNN edges, composite loss). Two switches for the
# diagnostics: PairNorm on/off, physics prior on/off.
#
# THE BUG FIX is in composite_loss: the loss is computed on `m` only
# (observed cells of the training split). The old code overwrote it with an
# unmasked loss over every node.
# ============================================================

class SkipGATLayer(nn.Module):
    def __init__(self, cin, cout, dropout=0.2, pairnorm=True):
        super().__init__()
        self.gat = GATConv(cin, cout, heads=1, add_self_loops=False, concat=False,
                           dropout=dropout)
        self.self_linear = nn.Linear(cin, cout)
        self.fusion = nn.Linear(cout * 2, cout)
        self.norm = PairNorm(scale=1.0) if pairnorm else None
        self.p = dropout

    def forward(self, x, edge_index, batch=None):
        ei, _ = dropout_edge(edge_index, p=self.p, training=self.training)
        self_emb = self.self_linear(x)
        out = torch.cat([self_emb, self.gat(x, ei)], dim=-1)
        out = F.leaky_relu(self.fusion(out), 0.2) + self_emb
        return self.norm(out, batch) if self.norm is not None else out


def _spatial_over_time(layer, x_seq, edge_index):
    """Apply the graph layer at every pass of the window in one batched call.
    PairNorm normalises within each pass (bvec), as in the original."""
    N, Tw, _ = x_seq.shape
    x_flat = x_seq.permute(1, 0, 2).reshape(Tw * N, -1)
    E = edge_index.shape[1]
    offs = (torch.arange(Tw, device=x_seq.device) * N).repeat_interleave(E)
    ei = edge_index.repeat(1, Tw) + offs
    bvec = torch.arange(Tw, device=x_seq.device).repeat_interleave(N)
    return layer(x_flat, ei, bvec).view(Tw, N, -1).permute(1, 0, 2).contiguous()


class DualHeadGAT(nn.Module):
    def __init__(self, n_feat, spatial_dim, lstm_hidden, dropout, pairnorm=True,
                 n_atmos=6, n_static=5):
        super().__init__()
        self.na, self.ns = n_atmos, n_static
        self.spatial = SkipGATLayer(n_feat, spatial_dim, dropout, pairnorm)
        self.lstm = nn.LSTM(spatial_dim, lstm_hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.physics_head = nn.Sequential(
            nn.Linear(lstm_hidden + n_atmos, lstm_hidden), nn.LayerNorm(lstm_hidden),
            nn.LeakyReLU(0.2), nn.Linear(lstm_hidden, 1))
        self.residual_head = nn.Sequential(
            nn.Linear(lstm_hidden + n_static, lstm_hidden // 2), nn.LayerNorm(lstm_hidden // 2),
            nn.LeakyReLU(0.2), nn.Linear(lstm_hidden // 2, 1))
        self.res_scale = nn.Parameter(torch.tensor([1.0]))

    def forward(self, x_seq, edge_index):
        h, _ = self.lstm(_spatial_over_time(self.spatial, x_seq, edge_index))
        last = self.drop(h[:, -1, :])
        xl = x_seq[:, -1, :]
        t_phys = self.physics_head(torch.cat([last, xl[:, :self.na]], -1)).squeeze(-1)
        t_res = self.residual_head(
            torch.cat([last, xl[:, self.na:self.na + self.ns]], -1)).squeeze(-1) * self.res_scale
        return t_phys + t_res, t_phys, t_res


def composite_loss(out, y, m, ei, raw_wind, raw_last, temp_std, prior=True):
    pred, t_phys, t_res = out
    if m.sum() < 1:
        z = torch.zeros((), device=pred.device, requires_grad=True)
        return z, z
    fit = F.huber_loss(pred[m], y[m], delta=HP['huber_delta'])     # <- masked. The fix.
    if not prior:
        return fit, fit
    row, col = ei
    if row.numel() == 0:
        physics = torch.zeros((), device=pred.device)
    else:
        ws = torch.sqrt(raw_wind[row, 0] ** 2 + raw_wind[row, 1] ** 2 + 1e-6)
        physics = torch.mean(torch.abs(t_phys[row] - t_phys[col]) / (ws + 1.0))
    urban = (raw_last[:, 9] > COVER_T).float()
    veg = (raw_last[:, 6] > NDVI_T).float()
    neutral = 1.0 - (urban + veg).clamp(0, 1)
    push_m, pull_m = PUSH_MARGIN_C / max(temp_std, 1e-6), PULL_MARGIN_C / max(temp_std, 1e-6)
    push = (F.relu(push_m - t_res) * urban).sum() / urban.sum().clamp(min=1.0)
    pull = (F.relu(t_res + pull_m) * veg).sum() / veg.sum().clamp(min=1.0)
    sparsity = (t_res ** 2 * neutral).sum() / neutral.sum().clamp(min=1.0)
    total = (fit + HP['alpha'] * physics + HP['gamma'] * (push + pull)
             + HP['sparsity_w'] * sparsity)
    return total, fit


def train_predict_gat(R, S, spec):
    """Train on training cells, early-stop on validation cells, predict every cell (°C)."""
    set_seed(HP['seed'])
    W, tgt, n_win = S['W'], S['tgt'], S['n_win']
    fmask = torch.ones(len(FEATURE_COLS), device=device)
    for i in spec['drop']:
        fmask[i] = 0.0                                 # z-scored 0 = training mean
    Xn = torch.tensor(S['x_norm'], device=device)
    Xr = torch.tensor(R['cube'], device=device)
    Yn = torch.tensor((R['y_filled'] - S['t_mean']) / (S['t_std'] + 1e-6),
                      dtype=torch.float32, device=device)
    M = {k: torch.tensor(v, device=device) for k, v in S['masks'].items()}
    pos = torch.tensor(R['pos'], dtype=torch.float32, device=device)
    base_ei = torch.tensor(knn_edge_index(R['pos'], HP['k_neighbors']), device=device)
    vecs = pos[base_ei[1]] - pos[base_ei[0]]

    def window(j):
        t = int(tgt[j])
        x = Xn[:, t - W + 1:t + 1, :] * fmask
        wind = Xr[:, t, 1:3].mean(0)                    # citywide mean u10, v10 (raw)
        ei = base_ei[:, (vecs * wind).sum(1) >= 0]      # wind-pruned edges
        return x, ei, Yn[:, t], Xr[:, t, 1:3], Xr[:, t, :]

    model = DualHeadGAT(len(FEATURE_COLS), HP['spatial_dim'], HP['lstm_hidden'],
                        HP['dropout'], pairnorm=spec['pairnorm']).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', factor=0.5, patience=8)
    train_j = [j for j in range(n_win) if M['train'][:, j].any()]
    val_j = [j for j in range(n_win) if M['val'][:, j].any()]
    rng = np.random.default_rng(HP['seed'])
    best, best_state, best_ep, since = np.inf, None, -1, 0
    t0 = time.time()
    for ep in range(HP['epochs']):
        model.train()
        for j in rng.permutation(train_j):
            x, ei, y, rw, rl = window(j)
            opt.zero_grad()
            loss, _ = composite_loss(model(x, ei), y, M['train'][:, j], ei, rw, rl,
                                     S['t_std'], prior=spec['prior'])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), HP['clip'])
            opt.step()
        model.eval(); sq, n = 0.0, 0
        with torch.no_grad():
            for j in val_j:
                x, ei, y, _, _ = window(j)
                m = M['val'][:, j]
                e = (model(x, ei)[0][m] - y[m]) * S['t_std']
                sq += float((e ** 2).sum()); n += int(m.sum())
        va = (sq / max(n, 1)) ** 0.5
        sched.step(va)
        if va < best - 1e-4:
            best, best_ep, since = va, ep + 1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
            if since >= HP['patience']:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    pred = np.zeros((R['ymat'].shape[0], n_win), np.float32)
    with torch.no_grad():
        for j in range(n_win):
            x, ei, _, _, _ = window(j)
            pred[:, j] = (model(x, ei)[0] * S['t_std'] + S['t_mean']).cpu().numpy()
    return pred, dict(val_rmse=best, best_epoch=best_ep, epochs_run=ep + 1,
                      minutes=(time.time() - t0) / 60, selected='fixed')


def train_predict_rf(R, S, spec):
    """RF on training cells; leaf size chosen on validation cells; predict every cell (°C)."""
    t0 = time.time()
    Dm = build_tree_design(R, S, spec['drop'], spec['lag'])
    y = R['y_filled'][:, S['tgt']]
    Xtr, ytr = Dm[S['masks']['train']], y[S['masks']['train']]
    Xva, yva = Dm[S['masks']['val']], y[S['masks']['val']]
    best, best_rmse, best_leaf = None, np.inf, None
    for leaf in RF['leaf_grid']:
        m = RandomForestRegressor(n_estimators=RF['n_estimators'], min_samples_leaf=leaf,
                                  max_features=RF['max_features'], n_jobs=-1,
                                  random_state=HP['seed']).fit(Xtr, ytr)
        r = float(np.sqrt(((m.predict(Xva) - yva) ** 2).mean()))
        if r < best_rmse:
            best, best_rmse, best_leaf = m, r, leaf
    pred = best.predict(Dm.reshape(-1, Dm.shape[-1])).reshape(y.shape).astype(np.float32)
    return pred, dict(val_rmse=best_rmse, best_epoch=-1, epochs_run=-1,
                      minutes=(time.time() - t0) / 60, selected=f'leaf{best_leaf}')

In [ ]:
# ============================================================
# 5. RUN  (resumable: finished runs are skipped)
# ============================================================
def pred_path(cfg, mode, fold, model):
    return f"{PRED_DIR}/{cfg['label']}_{mode}_f{fold}_{model}.npz"


def run_plan():
    log_path = f'{RES_DIR}/run_log.csv'
    log = pd.read_csv(log_path).to_dict('records') if os.path.exists(log_path) else []
    for stage, models, modes, folds in RUN_PLAN:
        if not folds:
            continue
        print(f'\n===== STAGE {stage}: {models} | {modes} | folds {folds} =====')
        for cfg in CONFIGS:
            R = load_raw(cfg)
            for mode in modes:
                for fold in folds:
                    S = None
                    for mname in models:
                        path = pred_path(cfg, mode, fold, mname)
                        if os.path.exists(path):
                            continue
                        if S is None:
                            S = make_split(R, mode, fold)
                            save_context(R, S)
                            print(f"  {cfg['label']} {mode} f{fold}: blocks tr/va/te "
                                  f"{S['n_blocks']}, nodes {S['n_nodes']}, "
                                  f"test cells {int(S['masks']['test'].sum())}")
                        spec = MODELS[mname]
                        fn = train_predict_gat if spec['kind'] == 'gat' else train_predict_rf
                        pred, info = fn(R, S, spec)
                        np.savez_compressed(path, pred=pred, **{k: np.array(v) for k, v in
                                                                info.items()})
                        log.append(dict(config=cfg['label'], mode=mode, fold=fold,
                                        model=mname, **info))
                        pd.DataFrame(log).to_csv(log_path, index=False)
                        print(f"    {mname:<15} val RMSE {info['val_rmse']:.3f}  "
                              f"[{info['selected']}, ep {info['best_epoch']}]  "
                              f"{info['minutes']:.1f} min")
    print('\nrun plan complete.')

run_plan()

In [ ]:
# ============================================================
# 6. METRICS  (everything below reads saved files only)
# ------------------------------------------------------------
# Per pass t (a pass is scored if >= MIN_NODES_PER_PASS test nodes observed):
#   w_t      nodes scored            ym_t, pm_t  observed / predicted mean
#   sq_t     sum of squared ANOMALY errors (each field minus its own mean)
#   sdo_t    observed spatial sd     sdp_t       predicted spatial sd
# Then, exactly:  MSE = sum w (pm-ym)^2 / sum w  +  sum sq / sum w
# Oracle ("structureless") reference = each pass's OBSERVED TEST-NODE mean:
#   its level error is 0 and its anomaly MSE is sum w sdo^2 / sum w.
# R2_sp = 1 - MSE_anom / MSE_oracle.   S = pooled sdp / pooled sdo.
# ============================================================
from functools import lru_cache


@lru_cache(maxsize=None)
def load_ctx(label, mode, fold):
    z = np.load(f'{CTX_DIR}/{label}_{mode}_f{fold}.npz', allow_pickle=True)
    return {k: z[k] for k in z.files}


def load_pred(label, mode, fold, model):
    p = f'{PRED_DIR}/{label}_{mode}_f{fold}_{model}.npz'
    return np.load(p)['pred'] if os.path.exists(p) else None


def have(label, mode, fold, *models):
    return (os.path.exists(f'{CTX_DIR}/{label}_{mode}_f{fold}.npz') and
            all(os.path.exists(f'{PRED_DIR}/{label}_{mode}_f{fold}_{m}.npz') for m in models))


def pass_stats(y, pred, mask):
    rows = []
    for t in np.flatnonzero(mask.sum(0) >= MIN_NODES_PER_PASS):
        m = mask[:, t]; a = y[m, t].astype(np.float64); b = pred[m, t].astype(np.float64)
        aa, bb = a - a.mean(), b - b.mean()
        r = (np.corrcoef(a, b)[0, 1] if a.std() > 1e-9 and b.std() > 1e-9 else np.nan)
        rows.append((t, m.sum(), a.mean(), b.mean(), ((bb - aa) ** 2).sum(),
                     a.std(), b.std(), r))
    cols = ['t', 'w', 'ym', 'pm', 'sq', 'sdo', 'sdp', 'r']
    return pd.DataFrame(rows, columns=cols)


def summarise(ps, t2m_level=None):
    """Aggregate pass_stats into the paper's metrics."""
    if len(ps) == 0:
        return {}
    w = ps.w.values.astype(float); W_ = w.sum()
    lev = ps.pm.values - ps.ym.values
    mse_l = (w * lev ** 2).sum() / W_
    mse_a = ps.sq.sum() / W_
    mse_o = (w * ps.sdo ** 2).sum() / W_
    S = np.sqrt((w * ps.sdp ** 2).sum() / (w * ps.sdo ** 2).sum())
    r = np.nanmean(ps.r.values)
    out = dict(rmse=np.sqrt(mse_l + mse_a), level=np.sqrt(mse_l), anom=np.sqrt(mse_a),
               oracle=np.sqrt(mse_o), r2sp=1 - mse_a / mse_o, S=S, r_sp=r, S_over_r=S / r,
               level_bias=lev.mean(), level_sd=lev.std(),
               frac_systematic=lev.mean() ** 2 / max(lev.mean() ** 2 + lev.std() ** 2, 1e-12),
               obs_level_sd=ps.ym.std(), n_passes=len(ps))
    if t2m_level is not None:
        tl = t2m_level[ps.t.values]
        out['corr_levelerr_t2m'] = (np.corrcoef(lev, tl)[0, 1] if lev.std() > 1e-9 else np.nan)
        out['corr_levelerr_obslevel'] = (np.corrcoef(lev, ps.ym)[0, 1]
                                         if lev.std() > 1e-9 else np.nan)
    return out


def scores(label, mode, fold, model, subset=None):
    """subset: optional boolean over windows (e.g. one sensor)."""
    ctx = load_ctx(label, mode, fold)
    pred = load_pred(label, mode, fold, model)
    if pred is None:
        return None
    mask = ctx['m_test'].copy()
    if subset is not None:
        mask &= subset[None, :]
    ps = pass_stats(ctx['y'], pred, mask)
    s = summarise(ps, ctx['t2m_level'])
    err = pred[mask] - ctx['y'][mask]
    s['rmse_all_cells'] = float(np.sqrt((err ** 2).mean()))
    s['n_nights'] = len(np.unique(ctx['night_id'][ps.t.values])) if len(ps) else 0
    return s


def bootstrap_by_night(label, mode, fold, m_tree=TREE, m_graph=GRAPH, n_boot=2000, seed=0):
    """Paired bootstrap over calendar nights (both passes of a night move together).
    Returns point gaps and 95% percentile intervals: graph minus tree for RMSE and
    anomaly RMSE, tree minus graph for R2_sp (so positive = tree better everywhere)."""
    ctx = load_ctx(label, mode, fold)
    mask = ctx['m_test']
    pt = pass_stats(ctx['y'], load_pred(label, mode, fold, m_tree), mask)
    pg = pass_stats(ctx['y'], load_pred(label, mode, fold, m_graph), mask)
    nid = ctx['night_id'][pt.t.values]
    nights, inv = np.unique(nid, return_inverse=True)
    K = len(nights)
    def agg(ps, col):
        return np.bincount(inv, weights=ps[col].values, minlength=K)
    w = agg(pt, 'w')
    tl, gl = agg(pt.assign(x=pt.w * (pt.pm - pt.ym) ** 2), 'x'), agg(pg.assign(x=pg.w * (pg.pm - pg.ym) ** 2), 'x')
    ta, ga = agg(pt, 'sq'), agg(pg, 'sq')
    oa = agg(pt.assign(x=pt.w * pt.sdo ** 2), 'x')

    def gaps(idx):
        Ws = w[idx].sum()
        t_rmse = np.sqrt((tl[idx].sum() + ta[idx].sum()) / Ws)
        g_rmse = np.sqrt((gl[idx].sum() + ga[idx].sum()) / Ws)
        t_a, g_a = np.sqrt(ta[idx].sum() / Ws), np.sqrt(ga[idx].sum() / Ws)
        o = oa[idx].sum() / Ws
        return g_rmse - t_rmse, g_a - t_a, (1 - t_a ** 2 / o) - (1 - g_a ** 2 / o)

    rng = np.random.default_rng(seed)
    point = gaps(np.arange(K))
    draws = np.array([gaps(rng.integers(0, K, K)) for _ in range(n_boot)])
    out = dict(n_nights=K)
    for i, name in enumerate(['rmse_gap', 'anom_gap', 'r2sp_gap']):
        out[name] = point[i]
        out[name + '_lo'] = np.percentile(draws[:, i], 2.5)
        out[name + '_hi'] = np.percentile(draws[:, i], 97.5)
        out[name + '_p_gt0'] = float((draws[:, i] > 0).mean())
    return out


LABELS = [c['label'] for c in CONFIGS]
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
R3 = lambda df: df.round(3)

In [ ]:
# ============================================================
# 7. RESULTS
# ============================================================
# ---- 7a. Table I: tree vs graph vs oracle, per mode, fold 0; and fold ranges ----
rows = []
for mode in MODES:
    for fold in FOLDS:
        for lab in LABELS:
            if not have(lab, mode, fold, TREE, GRAPH):
                continue
            t, g = scores(lab, mode, fold, TREE), scores(lab, mode, fold, GRAPH)
            rows.append(dict(mode=mode, fold=fold, config=lab,
                             tree_rmse=t['rmse'], tree_level=t['level'], tree_anom=t['anom'],
                             graph_rmse=g['rmse'], graph_level=g['level'], graph_anom=g['anom'],
                             oracle_rmse=t['oracle'], tree_r2sp=t['r2sp'], graph_r2sp=g['r2sp'],
                             n_passes=t['n_passes'], n_nights=t['n_nights'],
                             tree_rmse_all=t['rmse_all_cells'], graph_rmse_all=g['rmse_all_cells']))
main = pd.DataFrame(rows)
main['rmse_gap'] = main.graph_rmse - main.tree_rmse
main['level_gap'] = main.graph_level - main.tree_level
main['anom_gap'] = main.graph_anom - main.tree_anom
main['r2sp_gap'] = main.tree_r2sp - main.graph_r2sp
main['level_over_anom'] = np.where(main.anom_gap > 0, main.level_gap / main.anom_gap, np.nan)  # only meaningful when anom gap > 0
main.to_csv(f'{RES_DIR}/table1_all_folds.csv', index=False)

for mode in MODES:
    print(f'\n=== TABLE I — {mode}, fold 0 ===')
    print(R3(main[(main['mode'] == mode) & (main.fold == 0)].drop(columns=['mode', 'fold'])).to_string(index=False))
if main.fold.nunique() > 1:
    print('\n=== ACROSS FOLDS: mean [min, max] ===')
    agg = main.groupby(['mode', 'config'])[['tree_rmse', 'graph_rmse', 'tree_level', 'graph_level',
                                           'anom_gap', 'r2sp_gap']].agg(['mean', 'min', 'max'])
    print(R3(agg).to_string())
    pos = main.groupby('mode').apply(lambda d: f"{(d.anom_gap > 0).sum()}/{len(d)}")
    print('\nanomaly gap > 0 (tree better spatially) in config-folds:', dict(pos))

# ---- 7b. Decoupling: bootstrap by night, and do RMSE gaps order the spatial gaps? ----
bs_rows = []
for mode in MODES:
    for fold in FOLDS:
        for lab in LABELS:
            if have(lab, mode, fold, TREE, GRAPH):
                bs_rows.append(dict(mode=mode, fold=fold, config=lab,
                                    **bootstrap_by_night(lab, mode, fold)))
boot = pd.DataFrame(bs_rows)
boot.to_csv(f'{RES_DIR}/bootstrap_by_night.csv', index=False)
print('\n=== BOOTSTRAP (calendar nights, 2000 draws, 95%) — fold 0 ===')
cols = ['config', 'n_nights', 'rmse_gap', 'rmse_gap_lo', 'rmse_gap_hi', 'anom_gap', 'anom_gap_lo',
        'anom_gap_hi', 'r2sp_gap', 'r2sp_gap_lo', 'r2sp_gap_hi']
for mode in MODES:
    b = boot[(boot['mode'] == mode) & (boot.fold == 0)]
    print(f'\n-- {mode}'); print(R3(b[cols]).to_string(index=False))
    print(f"   95% interval excludes zero (anomaly gap): {(b.anom_gap_lo > 0).sum()}/{len(b)}; "
          f"(R2_sp gap): {(b.r2sp_gap_lo > 0).sum()}/{len(b)}")


def order_disagreements(a, b):
    """Pairs (i, j) where a and b order the two configs differently."""
    out = []
    n = len(a)
    for i in range(n):
        for j in range(i + 1, n):
            if (a[i] - a[j]) * (b[i] - b[j]) < 0:
                out.append((i, j))
    return out

print('\n=== DECOUPLING: does the RMSE gap order the spatial gaps? (fold 0) ===')
for mode in MODES:
    d = main[(main['mode'] == mode) & (main.fold == 0)].reset_index(drop=True)
    if len(d) < 3:
        continue
    for target in ('anom_gap', 'r2sp_gap'):
        dis = order_disagreements(d.rmse_gap.values, d[target].values)
        rho = stats.spearmanr(d.rmse_gap, d[target]).statistic
        print(f"  {mode:<15} RMSE gap vs {target:<9}: Spearman {rho:+.2f}; "
              f"{len(dis)}/{len(d) * (len(d) - 1) // 2} config pairs ordered differently: "
              + ', '.join(f"{d.config[i]}~{d.config[j]}" for i, j in dis))
    print(f"  {mode:<15} level gap / anomaly gap: {d.level_over_anom.min():.1f}"
          f"–{d.level_over_anom.max():.1f}")

# ---- 7c. Ablation of static surface predictors, with the decomposition ----
ab_rows = []
for mode in MODES:
    for fold in FOLDS:
        for lab in LABELS:
            for full, abl in (('RF_lag', 'RF_noStatic'), ('GAT', 'GAT_noStatic')):
                if not have(lab, mode, fold, full, abl):
                    continue
                a, b = scores(lab, mode, fold, full), scores(lab, mode, fold, abl)
                ab_rows.append(dict(mode=mode, fold=fold, config=lab, model=full,
                                    rmse_full=a['rmse'], rmse_abl=b['rmse'],
                                    level_full=a['level'], level_abl=b['level'],
                                    anom_full=a['anom'], anom_abl=b['anom'],
                                    r2sp_full=a['r2sp'], r2sp_abl=b['r2sp'],
                                    rsp_full=a['r_sp'], rsp_abl=b['r_sp']))
abl = pd.DataFrame(ab_rows)
if len(abl):
    abl['d_rmse'] = abl.rmse_abl - abl.rmse_full          # >0: ablation hurts RMSE
    abl['d_r2sp'] = abl.r2sp_full - abl.r2sp_abl          # >0: ablation hurts spatial skill
    abl['d_rsp'] = abl.rsp_full - abl.rsp_abl
    abl['rank_reversal'] = (abl.d_rmse < 0) & (abl.d_r2sp > 0)   # RMSE prefers the ablated model
    abl.to_csv(f'{RES_DIR}/ablation_static.csv', index=False)
    print('\n=== ABLATION: remove static surface predictors (fold 0) ===')
    for (mode, model), d in abl[abl.fold == 0].groupby(['mode', 'model']):
        print(f'\n-- {model}, {mode}')
        print(R3(d[['config', 'rmse_full', 'rmse_abl', 'level_full', 'level_abl', 'anom_full',
                    'anom_abl', 'r2sp_full', 'r2sp_abl', 'rsp_full', 'rsp_abl',
                    'rank_reversal']]).to_string(index=False))
        print(f"   spatial r drops in {(d.d_rsp > 0).sum()}/{len(d)}, mean "
              f"{d.rsp_full.mean():.3f} -> {d.rsp_abl.mean():.3f}; |dRMSE| max "
              f"{d.d_rmse.abs().max():.3f}; RMSE prefers ablated in {(d.d_rmse < 0).sum()}/{len(d)}; "
              f"rank reversals {d.rank_reversal.sum()}/{len(d)}")

# ---- 7d. GAT diagnostics: why is its level error large? ----
dg_rows = []
for mode in MODES:
    for lab in LABELS:
        for m in ('GAT', 'GAT_noPairNorm', 'GAT_noPrior', 'RF_lag'):
            if have(lab, mode, 0, m):
                s = scores(lab, mode, 0, m)
                dg_rows.append(dict(mode=mode, config=lab, model=m, **{k: s[k] for k in (
                    'rmse', 'level', 'anom', 'r2sp', 'level_bias', 'level_sd', 'frac_systematic',
                    'corr_levelerr_t2m', 'corr_levelerr_obslevel', 'obs_level_sd')}))
diag = pd.DataFrame(dg_rows)
diag.to_csv(f'{RES_DIR}/gat_diagnostics.csv', index=False)
print('\n=== GAT DIAGNOSTICS (fold 0) ===')
print('level_sd is the night-to-night sd of the SIGNED level error; obs_level_sd is the sd of the')
print('OBSERVED citywide mean across passes (the signal). corr_levelerr_* < 0 means the model')
print('under-follows night-to-night swings (too cool on warm nights, too warm on cool ones).')
for mode, d in diag.groupby('mode'):
    print(f'\n-- {mode}')
    print(R3(d.drop(columns='mode')).to_string(index=False))

# ---- 7e. Spread ratio with S/r ----
sp_rows = []
for mode in MODES:
    for lab in LABELS:
        for m in ('RF_lag', 'RF_plain', 'GAT'):
            if have(lab, mode, 0, m):
                s = scores(lab, mode, 0, m)
                sp_rows.append(dict(mode=mode, config=lab, model=m, S=s['S'], r_sp=s['r_sp'],
                                    S_over_r=s['S_over_r']))
spread = pd.DataFrame(sp_rows)
spread.to_csv(f'{RES_DIR}/spread.csv', index=False)
print('\n=== SPREAD RATIO S, spatial r, S/r (fold 0) ===')
print('For a calibrated conditional-mean predictor S ~ r, so S/r ~ 1. S/r < 1 = flatter than')
print('expected; S/r > 1 = more spread than its skill supports.')
print(R3(spread.pivot_table(index=['mode', 'config'], columns='model',
                            values=['S', 'r_sp', 'S_over_r'])).to_string())
for mode in MODES:
    d = spread[spread['mode'] == mode].pivot_table(index='config', columns='model', values=['S', 'r_sp'])
    if ('S', 'RF_lag') in d and ('S', 'RF_plain') in d:
        print(f"  {mode}: spatial lag raises S in {(d[('S','RF_lag')] > d[('S','RF_plain')]).sum()}/{len(d)}, "
              f"raises r in {(d[('r_sp','RF_lag')] > d[('r_sp','RF_plain')]).sum()}/{len(d)}")

# ---- 7f. Per sensor (Terra vs Aqua) ----
se_rows = []
for mode in MODES:
    for lab in LABELS:
        if not have(lab, mode, 0, TREE, GRAPH):
            continue
        sen = load_ctx(lab, mode, 0)['sensor']
        for s_name in ('Terra', 'Aqua'):
            sub = sen == s_name
            if sub.sum() == 0:
                continue
            t, g = scores(lab, mode, 0, TREE, sub), scores(lab, mode, 0, GRAPH, sub)
            if not t or not g:
                continue
            se_rows.append(dict(mode=mode, config=lab, sensor=s_name, n_passes=t['n_passes'],
                                tree_rmse=t['rmse'], tree_anom=t['anom'], graph_rmse=g['rmse'],
                                graph_anom=g['anom'], oracle=t['oracle'],
                                anom_gap=g['anom'] - t['anom']))
sens = pd.DataFrame(se_rows)
sens.to_csv(f'{RES_DIR}/per_sensor.csv', index=False)
print('\n=== PER SENSOR (fold 0) ===')
print(R3(sens).to_string(index=False) if len(sens) else '  sensor labels unavailable')
if len(sens):
    for mode, d in sens.groupby('mode'):
        print(f"  {mode}: tree better spatially in {(d.anom_gap > 0).sum()}/{len(d)} sensor-configs")

# ---- 7g. Split design: does the tree's small level error survive unseen nights? ----
if set(MODES) <= set(main['mode']):
    f0 = main[main.fold == 0].pivot_table(index='config', columns='mode',
                                          values=['tree_level', 'graph_level', 'anom_gap', 'r2sp_gap'])
    print('\n=== SPATIAL vs SPATIOTEMPORAL (fold 0) — referee comment 1 ===')
    print(R3(f0).to_string())

In [ ]:
# ============================================================
# 8. DATA CHECKS — the open doubts, answered from the data
# ============================================================
chk = []          # one row per config, collected into data_checks.csv

def moran_I(values, pos, k=8, n_perm=999, seed=0):
    """Moran's I with binary kNN weights (row-standardised) and a permutation p-value."""
    z = values - values.mean()
    nbr = knn_indices(pos, k)
    def I(zz):
        return (len(zz) / (len(zz) * 1.0)) * (zz * zz[nbr].mean(1)).sum() / (zz ** 2).sum()
    obs = I(z)
    rng = np.random.default_rng(seed)
    perm = np.array([I(rng.permutation(z)) for _ in range(n_perm)])
    return obs, (1 + (perm >= obs).sum()) / (n_perm + 1)


for cfg in CONFIGS:
    lab = cfg['label']
    R = load_raw(cfg)
    y, obs, cube = R['ymat'], R['observed'], R['cube']
    N, T = y.shape
    row = dict(config=lab)

    # --- D1: which static predictor tracks night LST? (paper: "+0.44 to +0.69") ---
    for f in ('bld_cover', 'NDBI', 'NDVI', 'bld_height'):
        fi = FEATURE_COLS.index(f)
        rs = [np.corrcoef(cube[obs[:, t], t, fi], y[obs[:, t], t])[0, 1]
              for t in range(T) if obs[:, t].sum() >= 30 and cube[obs[:, t], t, fi].std() > 0]
        row[f'corr_{f}_perpass'] = np.nanmean(rs)
        row[f'corr_{f}_pooled'] = np.corrcoef(cube[:, :, fi][obs], y[obs])[0, 1]

    # --- D2: variability. Signal level sd vs spatial sd (Introduction) ---
    pm = np.array([y[obs[:, t], t].mean() if obs[:, t].any() else np.nan for t in range(T)])
    good = np.array([obs[:, t].sum() >= MIN_NODES_PER_PASS for t in range(T)])
    row['total_sd_obs'] = float(np.nanstd(y[obs]))
    row['level_sd_allnodes'] = float(np.nanstd(pm[good]))
    wv = np.array([obs[:, t].sum() for t in range(T)])[good]
    vv = np.array([y[obs[:, t], t].var() for t in range(T) if good[t]])
    row['spatial_sd_pooled_allnodes'] = float(np.sqrt((wv * vv).sum() / wv.sum()))
    row['spatial_sd_old_definition'] = float(np.nanmean(np.nanstd(y, axis=0)))   # old header number
    sen = R['sensor']
    if (sen == 'Terra').any() and (sen == 'Aqua').any():
        row['terra_minus_aqua_level'] = float(np.nanmean(pm[good & (sen == 'Terra')]) -
                                              np.nanmean(pm[good & (sen == 'Aqua')]))
    if os.path.exists(f'{CTX_DIR}/{lab}_spatial_f0.npz'):
        c0 = load_ctx(lab, 'spatial', 0)
        test_nodes = c0['node_role'] == 'test'
        ps = pass_stats(c0['y'], c0['y'], c0['m_test'])        # pred = y: just the obs stats
        row['level_sd_testnodes'] = float(ps.ym.std())
        row['spatial_sd_pooled_testnodes'] = float(np.sqrt((ps.w * ps.sdo ** 2).sum() / ps.w.sum()))
    else:
        test_nodes = None

    # --- D3: persistence, defined properly: same sensor, previous calendar night, observed ---
    def persistence(node_sel):
        num, den = 0.0, 0
        for t in range(T):
            prev = np.flatnonzero((R['night_day'] == R['night_day'][t] - 1) & (sen == sen[t]))
            if len(prev) == 0:
                continue
            p = prev[0]
            m = obs[:, t] & obs[:, p] & node_sel
            num += float(((y[m, t] - y[m, p]) ** 2).sum()); den += int(m.sum())
        return np.sqrt(num / den) if den else np.nan
    row['persistence_allnodes'] = persistence(np.ones(N, bool))
    if test_nodes is not None:
        row['persistence_testnodes'] = persistence(test_nodes)
    yf = R['y_filled']
    row['persistence_old_definition'] = float(np.sqrt(np.mean((yf[:, 1:] - yf[:, :-1]) ** 2)))

    # --- D4: counts ---
    row.update(passes=T, terra=int((sen == 'Terra').sum()), aqua=int((sen == 'Aqua').sum()),
               nights=int(R['night_id'].max() + 1),
               max_passes_per_night=int(np.bincount(R['night_id']).max()),
               nodes_total=R['n_nodes_total'], nodes_kept=N,
               pct_target_interpolated=100 * float((~obs).mean()),
               blocks_occupied=len(np.unique(R['block'])))
    for mode in MODES:
        for fold in FOLDS:
            if os.path.exists(f'{CTX_DIR}/{lab}_{mode}_f{fold}.npz'):
                c = load_ctx(lab, mode, fold)
                row[f'{mode}_f{fold}_blocks_tr_va_te'] = '/'.join(map(str, c['n_blocks']))
                row[f'{mode}_f{fold}_nodes_tr_va_te'] = '/'.join(map(str, c['n_nodes']))
                row[f'{mode}_f{fold}_scored_passes'] = int((c['m_test'].sum(0) >= MIN_NODES_PER_PASS).sum())

    # --- D5: residual spatial autocorrelation (replaces "+0.544"): Moran's I ---
    if have(lab, 'spatial', 0, TREE):
        c0 = load_ctx(lab, 'spatial', 0)
        pr = load_pred(lab, 'spatial', 0, TREE)
        m = c0['m_test']
        res = np.where(m, pr - c0['y'], np.nan)
        node_mean = np.nanmean(res, axis=1)
        ok = np.isfinite(node_mean)
        I, p = moran_I(node_mean[ok], c0['pos'][ok])
        row['moran_I_tree_residual'] = I; row['moran_p'] = p; row['moran_n_nodes'] = int(ok.sum())

    # --- D6: NO2 temporal resolution (is it daily, or a composite?) ---
    no2 = cube[:, :, FEATURE_COLS.index('traffic_no2_proxy')]
    raw_no2 = R['df_night'].pivot_table(index=ID_COL, columns=TIME_COL,
                                        values='traffic_no2_proxy', aggfunc='first')
    row['no2_distinct_values_per_node_median'] = float(raw_no2.nunique(axis=1).median())
    row['no2_frac_variance_within_node'] = float(no2.var(axis=1).mean() / no2.var())

    # --- D7: columns available for view angle / QC ---
    row['view_angle_columns'] = ','.join(c for c in R['all_columns']
                                         if any(s in c.lower() for s in ('view', 'angle', 'zenith', 'vza')))
    row['qc_columns'] = json.dumps(R['qc_info'])
    chk.append(row)

checks = pd.DataFrame(chk).set_index('config')
checks.to_csv(f'{RES_DIR}/data_checks.csv')
print('=== DATA CHECKS (one column per config) ===')
print(checks.T.to_string())
print("""
How to read these:
  D1 corr_*_perpass : mean within-pass spatial correlation with night LST. Whichever
                      variable gives +0.44..+0.69 is the one the paper should name.
  D2 level_sd_*     : sd of the OBSERVED citywide mean across passes = the temporal signal.
                      Compare with spatial_sd_pooled_* for the Introduction's claim.
                      spatial_sd_old_definition reproduces the old 0.75-1.20 header number.
  D3 persistence_*  : same sensor, previous calendar night, observed cells only.
                      persistence_old_definition reproduces the old 2.34-3.47.
  D4 counts         : passes, Terra/Aqua, nights, kept nodes, blocks and nodes per split.
                      max_passes_per_night should be <= 2; if not, the night grouping is off.
  D5 moran_I        : spatial autocorrelation of the tree's time-mean test residual.
  D6 no2_*          : ~1 distinct value per node, or tiny within-node variance = a static
                      composite, not a nightly field. State the aggregation either way.
  D7                : whether view angle or QC fields exist in the extraction.
  Oracle            : its mean is over TEST nodes only, per pass (see pass_stats).
""")

In [ ]:
# ============================================================
# 9. FIGURE — RMSE gap vs anomaly gap (both in °C), 95% bars by night
# ============================================================
mpl = matplotlib
mpl.rcParams.update({'pdf.fonttype': 42, 'font.family': 'serif',
                     'font.serif': ['Nimbus Roman', 'Liberation Serif', 'Times New Roman', 'DejaVu Serif'],
                     'font.size': 7, 'axes.linewidth': 0.6, 'mathtext.fontset': 'stix'})
COL = {'Blr': '#0072B2', 'Hyd': '#D55E00', 'Dlh': '#009E73'}

for mode in MODES:
    b = boot[(boot['mode'] == mode) & (boot.fold == 0)] if len(boot) else boot
    if len(b) == 0:
        continue
    fig, ax = plt.subplots(figsize=(3.2, 1.9), layout='constrained')
    for _, r in b.iterrows():
        c = COL.get(r.config[:3], 'k'); pre = r.config.endswith('Pre')
        ax.errorbar(r.rmse_gap, r.anom_gap,
                    yerr=[[r.anom_gap - r.anom_gap_lo], [r.anom_gap_hi - r.anom_gap]],
                    xerr=[[r.rmse_gap - r.rmse_gap_lo], [r.rmse_gap_hi - r.rmse_gap]],
                    fmt='o', ms=4.5, color=c, ecolor=c, elinewidth=0.7, capsize=1.2,
                    mfc=c if pre else 'white', mew=0.9)
        ax.annotate(r.config, (r.rmse_gap, r.anom_gap), xytext=(5, 3),
                    textcoords='offset points', fontsize=6, color=c)
    ax.axhline(0, color='0.6', lw=0.5)
    ax.set_xlabel('RMSE gap, graph $-$ tree ($^{\\circ}$C)')
    ax.set_ylabel('Anomaly-RMSE gap ($^{\\circ}$C)')
    ax.grid(alpha=0.25, lw=0.4)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    fig.savefig(f'{RES_DIR}/fig_decoupling_{mode}.pdf')
    fig.savefig(f'{RES_DIR}/fig_decoupling_{mode}.png', dpi=200)
    plt.close(fig)
    print(f'wrote {RES_DIR}/fig_decoupling_{mode}.pdf  (label offsets may need nudging)')

In [ ]:
# ============================================================
# 7h. GAT SETTINGS SENSITIVITY CHECK  (~20-25 min on a T4)
# ------------------------------------------------------------
# Question: do the paper's conclusions survive a different set of GAT settings?
# This is a sensitivity check, NOT a full tuning study: two alternative settings,
# one seed, fold 0. Every setting is scored on the SAME test cells as the main run.
#
#   GATs1: slower learning rate           (lr 3e-4,  everything else unchanged)
#   GATs2: stronger regularisation        (dropout 0.4, weight decay 1e-3)
#
# Time (from the earlier run log): same-night GAT ~2.4 min per run, unseen-night
# ~0.5 min. Defaults below: same-night on 3 configs, unseen-night on all 6
# -> about 2 x (3 x 2.4 + 6 x 0.5) = ~21 min. Set SWEEP_CONFIGS['spatial'] = None
# to run all six same-night configs (~35 min).
#
# Resumable: finished runs are skipped. Results -> results/gat_sensitivity.csv
# ============================================================
# ---- run order for this cell (NO retraining of the main models) ----
# 1. In section 1 set SMOKE_TEST = False (it was True for the smoke test).
# 2. Run sections 1, 2, 3, 4 and 6.  SKIP section 5: it would start training
#    folds 1-3, which takes hours. Sections 7-10 are not needed for this cell.
# 3. Run this cell.
if SMOKE_TEST:
    print('WARNING: SMOKE_TEST is True -> this uses the smoke-test folder and 3 epochs. '
          'Results are only a code check. Set SMOKE_TEST = False and re-run sections 1-4 and 6.')
if 'pred_path' not in globals():              # defined in section 5, which we skip
    def pred_path(cfg, mode, fold, model):
        return f"{PRED_DIR}/{cfg['label']}_{mode}_f{fold}_{model}.npz"

SWEEP = {
    'GATs1': dict(lr=3e-4),
    'GATs2': dict(dropout=0.4, weight_decay=1e-3),
}
SWEEP_CONFIGS = {'spatial': ['Blr-Pre', 'Hyd-Pre', 'Dlh-Pre'],   # None = all configs
                 'spatiotemporal': None}


def run_sweep():
    for name, over in SWEEP.items():
        for cfg in CONFIGS:
            R = None
            for mode in MODES:
                subset = SWEEP_CONFIGS.get(mode)
                if subset is not None and cfg['label'] not in subset:
                    continue
                path = pred_path(cfg, mode, 0, name)
                if os.path.exists(path):
                    continue
                if R is None:
                    R = load_raw(cfg)
                S = make_split(R, mode, 0)
                save_context(R, S)
                old = {k: HP[k] for k in over}
                HP.update(over)
                try:
                    pred, info = train_predict_gat(R, S, MODELS['GAT'])
                finally:
                    HP.update(old)                     # always restore the main settings
                np.savez_compressed(path, pred=pred, **{k: np.array(v) for k, v in info.items()})
                print(f"  {name} {cfg['label']} {mode}: val RMSE {info['val_rmse']:.3f} "
                      f"[best epoch {info['best_epoch']}/{info['epochs_run']}]  {info['minutes']:.1f} min")

run_sweep()

# ---- compare: does anything the paper claims change? ----
variants = ['GAT'] + list(SWEEP)
sens = []
for mode in MODES:
    for lab in LABELS:
        if not have(lab, mode, 0, TREE, 'GAT'):
            continue
        t = scores(lab, mode, 0, TREE)
        for v in variants:
            if not have(lab, mode, 0, v):
                continue
            g = scores(lab, mode, 0, v)
            val = float(np.load(pred_path(dict(label=lab), mode, 0, v))['val_rmse'])
            sens.append(dict(mode=mode, config=lab, variant=v, val_rmse=val,
                             gat_rmse=g['rmse'], gat_level=g['level'], gat_anom=g['anom'],
                             tree_rmse=t['rmse'], tree_level=t['level'], tree_anom=t['anom'],
                             anom_gap=g['anom'] - t['anom'], rmse_gap=g['rmse'] - t['rmse'],
                             level_gt_anom=g['level'] > g['anom'], gat_rmse_gt_oracle=g['rmse'] > g['oracle']))
sens = pd.DataFrame(sens)
sens.to_csv(f'{RES_DIR}/gat_sensitivity.csv', index=False)
print('\n=== GAT SETTINGS SENSITIVITY (fold 0) ===')
print('variant GAT = the settings used in the paper. anom_gap > 0: forest has lower anomaly error.')
for mode in MODES:
    d = sens[sens['mode'] == mode]
    if len(d):
        print(f'\n-- {mode}')
        print(R3(d.drop(columns='mode')).to_string(index=False))

print('\n=== WHAT TO CHECK ===')
for mode in MODES:
    d = sens[sens['mode'] == mode]
    for v in variants:
        dv = d[d.variant == v]
        if len(dv):
            print(f"{mode:<15} {v:<6}: forest lower anomaly error in {(dv.anom_gap > 0).sum()}/{len(dv)}; "
                  f"GAT level > its anomaly in {int(dv.level_gt_anom.sum())}/{len(dv)}; "
                  f"anomaly gap {dv.anom_gap.min():.3f} to {dv.anom_gap.max():.3f}; "
                  f"RMSE gap {dv.rmse_gap.min():.2f} to {dv.rmse_gap.max():.2f}")
    # the setting that validation RMSE would have chosen, per config
    best = d.loc[d.groupby('config').val_rmse.idxmin()]
    print(f"{mode:<15} validation-selected: forest lower anomaly error in "
          f"{(best.anom_gap > 0).sum()}/{len(best)}; variants chosen: {({k: int(v) for k, v in best.variant.value_counts().items()})}")
print("""
How to read this:
  * If the forest still has lower anomaly error and the GAT's level error still exceeds its
    anomaly error in the same-night rows, for all variants, the conclusions are robust to the
    settings. A Limitations sentence like "a small learning-rate/regularisation sweep did not
    change the conclusions" is then TRUE and can be added.
  * If any variant flips a headline (e.g. GAT beats the forest on anomaly error, or its level
    error drops below its anomaly error), the paper's numbers need updating before submission.
  * The PairNorm result is not re-tested here; it was measured at the fixed settings.
""")

In [ ]:
# ============================================================
# 10. PAPER NUMBERS — everything the text quotes, in one place
# ============================================================
def rng_(s, fmt='{:.2f}'):
    s = pd.Series(s).dropna()
    return (fmt + '–' + fmt).format(s.min(), s.max()) if len(s) else 'n/a'

lines = []
P = lines.append
for mode in MODES:
    d = main[(main['mode'] == mode) & (main.fold == 0)]
    if len(d) == 0:
        continue
    b = boot[(boot['mode'] == mode) & (boot.fold == 0)]
    P(f'\n######## {mode.upper()} split, fold 0 ########')
    P(f'Tree level RMSE {rng_(d.tree_level)} °C; graph level RMSE {rng_(d.graph_level)} °C')
    P(f'Graph level > its own anomaly in {(d.graph_level > d.graph_anom).sum()}/{len(d)}; '
      f'tree level < its anomaly in {(d.tree_level < d.tree_anom).sum()}/{len(d)}')
    P(f'Anomaly gap {rng_(d.anom_gap, "{:.3f}")} °C; R2_sp gap {rng_(d.r2sp_gap, "{:.3f}")}; '
      f'level/anomaly gap ratio {rng_(d.level_over_anom, "{:.1f}")}')
    P(f'Tree better spatially in {(d.anom_gap > 0).sum()}/{len(d)}; 95% night-bootstrap excludes 0 '
      f'in {(b.anom_gap_lo > 0).sum()}/{len(b)} (anomaly), {(b.r2sp_gap_lo > 0).sum()}/{len(b)} (R2_sp)')
    P(f'Oracle RMSE {rng_(d.oracle_rmse)} °C; graph RMSE > oracle in '
      f'{(d.graph_rmse > d.oracle_rmse).sum()}/{len(d)}; tree R2_sp {rng_(d.tree_r2sp)}')
    P(f'Scored passes {rng_(d.n_passes, "{:.0f}")}; nights {rng_(d.n_nights, "{:.0f}")}')
    if len(d) >= 2:
        i_r, i_a = d.rmse_gap.idxmax(), d.anom_gap.idxmax()
        P(f'Largest RMSE gap: {d.config[i_r]} ({d.rmse_gap[i_r]:.2f}); largest anomaly gap: '
          f'{d.config[i_a]} ({d.anom_gap[i_a]:.3f})')
        dis = order_disagreements(d.rmse_gap.values, d.anom_gap.values)
        P(f'Config pairs where RMSE-gap order != anomaly-gap order: {len(dis)} of '
          f'{len(d) * (len(d) - 1) // 2}')
    if main.fold.nunique() > 1:
        dm = main[main['mode'] == mode]
        P(f'Across {dm.fold.nunique()} folds: anomaly gap > 0 in {(dm.anom_gap > 0).sum()}/{len(dm)} '
          f'config-folds; range {rng_(dm.anom_gap, "{:.3f}")} °C')
    if len(abl):
        for model in ('RF_lag', 'GAT'):
            a = abl[(abl['mode'] == mode) & (abl.fold == 0) & (abl.model == model)]
            if len(a):
                P(f'Ablation {model}: r_sp {a.rsp_full.mean():.3f} -> {a.rsp_abl.mean():.3f} '
                  f'(drops {(a.d_rsp > 0).sum()}/{len(a)}); R2_sp drops {(a.d_r2sp > 0).sum()}/{len(a)}; '
                  f'|dRMSE| max {a.d_rmse.abs().max():.3f}; RMSE prefers ablated '
                  f'{(a.d_rmse < 0).sum()}/{len(a)}; rank reversals {a.rank_reversal.sum()}/{len(a)}')
    s = spread[spread['mode'] == mode]
    if len(s):
        P(f'Spread S {rng_(s.S)} ; S/r {rng_(s.S_over_r)} (all models)')
    g = diag[(diag['mode'] == mode) & (diag.model == 'GAT')]
    if len(g):
        P(f'GAT level bias {rng_(g.level_bias)} °C, systematic share {rng_(100 * g.frac_systematic, "{:.0f}")}%, '
          f'level-error sd {rng_(g.level_sd)} °C, corr(level err, t2m) {rng_(g.corr_levelerr_t2m)}')
        for v in ('GAT_noPairNorm', 'GAT_noPrior'):
            gv = diag[(diag['mode'] == mode) & (diag.model == v)]
            if len(gv):
                P(f'  {v}: level RMSE {rng_(gv.level)} (GAT {rng_(g.level)}), anomaly {rng_(gv.anom)}')
if len(checks):
    P('\n######## DATA ########')
    P(f"Passes {rng_(checks.passes, '{:.0f}')}; nights {rng_(checks.nights, '{:.0f}')}; "
      f"kept nodes {rng_(checks.nodes_kept, '{:.0f}')} of {rng_(checks.nodes_total, '{:.0f}')}; "
      f"interpolated {rng_(checks.pct_target_interpolated, '{:.0f}')}%")
    P(f"Level sd (signal) {rng_(checks.level_sd_allnodes)} °C vs pooled spatial sd "
      f"{rng_(checks.spatial_sd_pooled_allnodes)} °C (all nodes)")
    for f in ('bld_cover', 'NDBI', 'NDVI'):
        P(f"corr({f}, night LST) per pass: {rng_(checks[f'corr_{f}_perpass'])}")
    P(f"Persistence (same sensor, previous night): {rng_(checks.persistence_allnodes)} °C")
    if 'moran_I_tree_residual' in checks:
        P(f"Moran's I of tree residual: {rng_(checks.moran_I_tree_residual)}")
    P(f"NO2 distinct values per node (median): {rng_(checks.no2_distinct_values_per_node_median, '{:.0f}')}")
text = '\n'.join(lines)
print(text)
open(f'{RES_DIR}/paper_numbers.txt', 'w').write(text)
print(f'\nAll results in {RES_DIR}. Send me: paper_numbers.txt, table1_all_folds.csv, '
      f'bootstrap_by_night.csv, ablation_static.csv, gat_diagnostics.csv, spread.csv, '
      f'per_sensor.csv, data_checks.csv, run_log.csv, and the fig_decoupling PDFs.')